# 05 — Combined Strategy

Тук комбинираме няколко прости сигнала: frequency score, gap score и recent activity score. Целта е прозрачен score layer.


In [ ]:
from pathlib import Path
from itertools import combinations
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"


def read_csv_safe(path, **kwargs):
    path = Path(path)
    if not path.exists():
        print(f"Missing file: {path}")
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)


def number_columns(df: pd.DataFrame) -> list[str]:
    return [col for col in ["n1", "n2", "n3", "n4", "n5", "n6"] if col in df.columns]


def parse_numbers_text(value) -> list[int]:
    if pd.isna(value):
        return []
    parts = str(value).replace(";", ",").replace("|", ",").split(",")
    nums = []
    for part in parts:
        part = part.strip()
        if part.isdigit():
            nums.append(int(part))
    return nums


def extract_ticket_numbers(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    if len(number_columns(df)) == 6:
        return df.copy()
    text_col = next((c for c in df.columns if "numbers" in c.lower() or "числа" in c.lower()), None)
    if text_col is None:
        return df.copy()
    out = df.copy()
    parsed = out[text_col].apply(parse_numbers_text)
    for i in range(6):
        out[f"n{i+1}"] = parsed.apply(lambda nums: nums[i] if len(nums) > i else np.nan)
    return out


def show_latest_draw(df: pd.DataFrame) -> None:
    if df.empty:
        print("No draw data loaded.")
        return
    latest = df.tail(1).iloc[0]
    nums = [int(latest[col]) for col in number_columns(df)]
    print("Rows:", len(df))
    print("Latest date:", latest.get("date", "n/a"))
    print("Draw number:", latest.get("draw_number", latest.get("draw_no", "n/a")))
    print("Numbers:", nums)


def file_status(paths):
    rows = []
    for path in paths:
        path = Path(path)
        rows.append({
            "file": str(path.relative_to(PROJECT_ROOT)) if str(path).startswith(str(PROJECT_ROOT)) else str(path),
            "exists": path.exists(),
            "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else 0,
        })
    return pd.DataFrame(rows)


draws = read_csv_safe(DATA_DIR / "historical_draws.csv")
normalized = read_csv_safe(DATA_DIR / "v40_normalized_draw_events.csv")
canonical = read_csv_safe(DATA_DIR / "v41_canonical_draw_events.csv")
show_latest_draw(draws)


## Build combined number score


In [ ]:
num_cols = number_columns(draws)
all_numbers = draws[num_cols].to_numpy().ravel()
frequency = pd.Series(all_numbers).value_counts().reindex(range(1, 50), fill_value=0)
frequency_score = (frequency - frequency.min()) / (frequency.max() - frequency.min())
gap_records = []
for number in range(1, 50):
    hit_indices = draws.index[draws[num_cols].eq(number).any(axis=1)].tolist()
    current_gap = len(draws) - 1 - hit_indices[-1] if hit_indices else len(draws)
    gap_records.append((number, current_gap))
gap = pd.Series(dict(gap_records))
gap_score = (gap - gap.min()) / (gap.max() - gap.min())
recent_window = min(100, len(draws))
recent_numbers = draws.tail(recent_window)[num_cols].to_numpy().ravel()
recent_frequency = pd.Series(recent_numbers).value_counts().reindex(range(1, 50), fill_value=0)
recent_score = (recent_frequency - recent_frequency.min()) / (recent_frequency.max() - recent_frequency.min())
combined = pd.DataFrame({
    "number": range(1, 50),
    "frequency_score": frequency_score.values,
    "gap_score": gap_score.values,
    "recent_score": recent_score.values,
})
combined["combined_score"] = 0.45 * combined["frequency_score"] + 0.35 * combined["gap_score"] + 0.20 * combined["recent_score"]
combined.sort_values("combined_score", ascending=False).head(15)


In [ ]:
top = combined.sort_values("combined_score", ascending=False).head(15)
ax = top.set_index("number")[["frequency_score", "gap_score", "recent_score", "combined_score"]].plot(kind="bar", figsize=(13, 5), title="Top combined strategy scores")
ax.set_xlabel("Number")
ax.set_ylabel("Score")
plt.show()


## Извод

Combined strategy е полезна, защото прави единна таблица от няколко сигнала. Но теглата са аналитичен избор и трябва да се сравняват с backtest.
